In [9]:
# ============================================
# COMPLETE SMART PLACEMENT SYSTEM - DARK THEME
# Background: Black | Highlights: Dark Blue
# Features: Admin Register, Delete Questions/Students
# ============================================

# Install packages
!pip install flask pandas openpyxl numpy flask-session pyngrok PyPDF2 -q

import os
import pandas as pd
import numpy as np
from datetime import datetime
import secrets
import hashlib
from flask import Flask, render_template_string, request, send_file, redirect, url_for, session
from flask_session import Session
import socket
import re
import PyPDF2

# ============================================
# FIND AVAILABLE PORT
# ============================================

def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('', 0))
        return s.getsockname()[1]

# ============================================
# NGROK SETUP
# ============================================

try:
    from pyngrok import ngrok
    ngrok.set_auth_token("3CLntZ3WchkNmkBC57tcfliSXaZ_7T6oozpPs79J4ys3qbhT7")
    ngrok.kill()
    NGROK_AVAILABLE = True
    print("✅ ngrok ready!")
except:
    NGROK_AVAILABLE = False
    print("⚠️ Using local URL only")

# ============================================
# INITIALIZE DATABASES
# ============================================

def init_databases():
    for folder in ['data', 'uploads']:
        if not os.path.exists(folder):
            os.makedirs(folder)

    # Admin database
    if os.path.exists('data/admins.csv'):
        admins_df = pd.read_csv('data/admins.csv')
    else:
        admins_df = pd.DataFrame(columns=['admin_id', 'username', 'password', 'email', 'created_date'])
        default_admin = pd.DataFrame([{
            'admin_id': 'ADM001',
            'username': 'admin',
            'password': hashlib.md5('admin123'.encode()).hexdigest(),
            'email': 'admin@placement.com',
            'created_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }])
        admins_df = pd.concat([admins_df, default_admin], ignore_index=True)
        admins_df.to_csv('data/admins.csv', index=False)

    # Students database
    if os.path.exists('data/students.csv'):
        students_df = pd.read_csv('data/students.csv')
        if 'hackathons' not in students_df.columns:
            students_df['hackathons'] = 0
            students_df.to_csv('data/students.csv', index=False)
    else:
        students_df = pd.DataFrame(columns=[
            'student_id', 'name', 'reg_no', 'email', 'phone', 'cgpa', 'department',
            'skills', 'projects', 'certifications', 'hackathons', 'resume_path', 'password', 'registration_date'
        ])
        students_df.to_csv('data/students.csv', index=False)

    # Questions database
    if os.path.exists('data/questions.csv'):
        questions_df = pd.read_csv('data/questions.csv')
    else:
        questions_df = pd.DataFrame(columns=[
            'question_id', 'question', 'option_a', 'option_b', 'option_c', 'option_d',
            'answer', 'category', 'difficulty', 'created_date'
        ])
        questions_df.to_csv('data/questions.csv', index=False)

    # Results database
    if os.path.exists('data/results.csv'):
        results_df = pd.read_csv('data/results.csv')
    else:
        results_df = pd.DataFrame(columns=[
            'student_id', 'name', 'reg_no', 'email', 'exam_type', 'exam_score',
            'project_score', 'certification_score', 'hackathon_score', 'final_score', 'status',
            'correct_answers', 'wrong_answers', 'total_questions', 'exam_date'
        ])
        results_df.to_csv('data/results.csv', index=False)

    # Settings database
    if os.path.exists('data/settings.csv'):
        settings_df = pd.read_csv('data/settings.csv')
    else:
        settings_df = pd.DataFrame([{
            'setting_name': 'passing_score',
            'setting_value': 80,
            'updated_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }])
        settings_df.to_csv('data/settings.csv', index=False)

    return students_df, questions_df, results_df, settings_df, admins_df

students_df, questions_df, results_df, settings_df, admins_df = init_databases()

# ============================================
# PDF TO MCQ CONVERTER
# ============================================

def extract_mcq_from_pdf(pdf_path):
    questions = []
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            text = ""
            for page in pdf_reader.pages:
                text += page.extract_text()

        lines = text.split('\n')
        current_q = None

        for line in lines:
            line = line.strip()
            if not line:
                continue

            match_num = re.match(r'^(\d+)[\.\)\:]\s+(.+)', line)
            if match_num:
                if current_q and current_q.get('question'):
                    if 'option_a' not in current_q:
                        current_q['option_a'] = 'Option A'
                        current_q['option_b'] = 'Option B'
                        current_q['option_c'] = 'Option C'
                        current_q['option_d'] = 'Option D'
                    if 'answer' not in current_q:
                        current_q['answer'] = 'A'
                    questions.append(current_q)

                current_q = {
                    'question_id': len(questions) + 1,
                    'question': match_num.group(2),
                }
                continue

            match_opt = re.match(r'^([A-D])[\.\)\:]\s+(.+)', line)
            if match_opt and current_q:
                opt_letter = match_opt.group(1)
                opt_text = match_opt.group(2)
                if opt_letter == 'A':
                    current_q['option_a'] = opt_text
                elif opt_letter == 'B':
                    current_q['option_b'] = opt_text
                elif opt_letter == 'C':
                    current_q['option_c'] = opt_text
                elif opt_letter == 'D':
                    current_q['option_d'] = opt_text
                continue

            match_ans = re.search(r'[Aa]nswer\s*:?\s*([A-D])', line)
            if match_ans and current_q:
                current_q['answer'] = match_ans.group(1)

        if current_q and current_q.get('question'):
            if 'option_a' not in current_q:
                current_q['option_a'] = 'Option A'
                current_q['option_b'] = 'Option B'
                current_q['option_c'] = 'Option C'
                current_q['option_d'] = 'Option D'
            if 'answer' not in current_q:
                current_q['answer'] = 'A'
            questions.append(current_q)

        if len(questions) == 0:
            questions = [
                {'question_id': 1, 'question': 'Sample Question 1', 'option_a': 'Yes', 'option_b': 'No', 'option_c': 'Maybe', 'option_d': 'Not Sure', 'answer': 'A'},
                {'question_id': 2, 'question': 'Sample Question 2', 'option_a': 'Option A', 'option_b': 'Option B', 'option_c': 'Option C', 'option_d': 'Option D', 'answer': 'B'},
            ]

        return questions
    except Exception as e:
        print(f"Error: {e}")
        return []

# ============================================
# FLASK APP
# ============================================

app = Flask(__name__)
app.secret_key = secrets.token_hex(32)
app.config['SESSION_TYPE'] = 'filesystem'
app.config['MAX_CONTENT_LENGTH'] = 50 * 1024 * 1024
Session(app)

# ============================================
# HTML TEMPLATES - DARK THEME
# ============================================

LAYOUT = '''
<!DOCTYPE html>
<html>
<head>
    <title>Placement System</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.1.3/dist/css/bootstrap.min.css" rel="stylesheet">
    <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0/css/all.min.css" rel="stylesheet">
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }

        body {
            background: #000000;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            min-height: 100vh;
            color: #ffffff;
        }

        /* Dark Theme Navbar */
        .navbar {
            background: #0a0a0a !important;
            border-bottom: 2px solid #1a3a6e;
            box-shadow: 0 2px 20px rgba(0,0,0,0.5);
            padding: 1rem 0;
        }

        .navbar-brand {
            font-weight: bold;
            color: #1e90ff !important;
            font-size: 1.5rem;
        }

        .navbar-brand:hover { color: #4169e1 !important; }
        .nav-link { color: #cccccc !important; transition: 0.3s; }
        .nav-link:hover { color: #1e90ff !important; }

        /* Cards */
        .card {
            background: #0a0a0a;
            border: 1px solid #1a3a6e;
            border-radius: 15px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.3);
            transition: transform 0.3s, box-shadow 0.3s;
            color: #ffffff;
        }

        .card:hover {
            transform: translateY(-5px);
            box-shadow: 0 15px 40px rgba(30,144,255,0.2);
            border-color: #1e90ff;
        }

        .card-header {
            background: linear-gradient(135deg, #0a1a3a, #0d1f44);
            border-bottom: 1px solid #1e90ff;
            border-radius: 15px 15px 0 0 !important;
            color: #1e90ff;
            font-weight: bold;
        }

        .card-body { color: #e0e0e0; }

        /* Buttons */
        .btn-primary {
            background: linear-gradient(135deg, #0a1a3a, #0d1f44);
            border: 1px solid #1e90ff;
            border-radius: 25px;
            padding: 10px 25px;
            font-weight: 600;
            color: #1e90ff;
            transition: all 0.3s;
        }

        .btn-primary:hover {
            background: linear-gradient(135deg, #1e90ff, #4169e1);
            border-color: #1e90ff;
            color: #ffffff;
            transform: translateY(-2px);
            box-shadow: 0 5px 20px rgba(30,144,255,0.3);
        }

        .btn-success {
            background: linear-gradient(135deg, #0a2a1a, #0d4420);
            border: 1px solid #00ff88;
            color: #00ff88;
        }

        .btn-success:hover {
            background: linear-gradient(135deg, #00ff88, #00cc66);
            color: #000000;
        }

        .btn-danger {
            background: linear-gradient(135deg, #2a0a0a, #440d0d);
            border: 1px solid #ff4444;
            color: #ff4444;
        }

        .btn-danger:hover {
            background: linear-gradient(135deg, #ff4444, #cc0000);
            color: #ffffff;
        }

        .btn-warning {
            background: linear-gradient(135deg, #2a2a0a, #44440d);
            border: 1px solid #ffaa00;
            color: #ffaa00;
        }

        .btn-warning:hover {
            background: linear-gradient(135deg, #ffaa00, #cc8800);
            color: #000000;
        }

        .btn-info {
            background: linear-gradient(135deg, #0a2a2a, #0d4444);
            border: 1px solid #00ccff;
            color: #00ccff;
        }

        .btn-info:hover {
            background: linear-gradient(135deg, #00ccff, #0099cc);
            color: #000000;
        }

        /* Stat Cards */
        .stat-card {
            background: linear-gradient(135deg, #0a1a3a, #0d1f44);
            border: 1px solid #1e90ff;
            border-radius: 15px;
            padding: 20px;
            text-align: center;
            transition: all 0.3s;
        }

        .stat-card:hover {
            transform: translateY(-5px);
            box-shadow: 0 10px 30px rgba(30,144,255,0.2);
            border-color: #1e90ff;
        }

        .stat-number {
            font-size: 2rem;
            font-weight: bold;
            color: #1e90ff;
        }

        .stat-card p { color: #cccccc; margin-top: 10px; }
        .stat-card i { color: #1e90ff; margin-bottom: 10px; }

        /* Form Controls */
        .form-control, .form-select {
            background: #1a1a1a;
            border: 1px solid #1a3a6e;
            color: #ffffff;
            border-radius: 10px;
        }

        .form-control:focus, .form-select:focus {
            background: #1a1a1a;
            border-color: #1e90ff;
            color: #ffffff;
            box-shadow: 0 0 0 0.25rem rgba(30,144,255,0.25);
        }

        .form-label { color: #1e90ff; font-weight: 500; }

        /* Tables */
        .table {
            color: #e0e0e0;
            border-color: #1a3a6e;
        }

        .table-striped > tbody > tr:nth-of-type(odd) > * {
            background-color: #0a0a0a;
            color: #e0e0e0;
        }

        .table-striped > tbody > tr:nth-of-type(even) > * {
            background-color: #0f0f0f;
            color: #e0e0e0;
        }

        .table thead th {
            background: #0a1a3a;
            color: #1e90ff;
            border-bottom: 2px solid #1e90ff;
        }

        /* Delete Button */
        .delete-btn {
            background: #2a0a0a;
            color: #ff4444;
            border: 1px solid #ff4444;
            border-radius: 5px;
            padding: 5px 10px;
            cursor: pointer;
            text-decoration: none;
            font-size: 0.85rem;
            transition: all 0.3s;
        }

        .delete-btn:hover {
            background: #ff4444;
            color: #ffffff;
        }

        /* Alerts */
        .alert-success {
            background: #0a2a1a;
            border: 1px solid #00ff88;
            color: #00ff88;
        }

        .alert-danger {
            background: #2a0a0a;
            border: 1px solid #ff4444;
            color: #ff4444;
        }

        .alert-warning {
            background: #2a2a0a;
            border: 1px solid #ffaa00;
            color: #ffaa00;
        }

        .alert-info {
            background: #0a2a2a;
            border: 1px solid #00ccff;
            color: #00ccff;
        }

        /* Footer */
        footer {
            background: #0a0a0a;
            border-top: 1px solid #1a3a6e;
            color: #666666;
            padding: 20px 0;
            margin-top: 50px;
            text-align: center;
        }

        /* Progress Bar */
        .progress {
            background: #1a1a1a;
            border-radius: 10px;
        }

        .progress-bar {
            background: linear-gradient(90deg, #1e90ff, #4169e1);
            border-radius: 10px;
        }

        /* Timer */
        .timer {
            background: #0a0a0a;
            border: 1px solid #1e90ff;
            color: #1e90ff;
            font-size: 1.2rem;
            font-weight: bold;
        }

        /* Links */
        a { color: #1e90ff; text-decoration: none; transition: 0.3s; }
        a:hover { color: #4169e1; }

        /* Fade In Animation */
        .fade-in {
            animation: fadeIn 0.5s ease-out;
        }

        @keyframes fadeIn {
            from { opacity: 0; transform: translateY(20px); }
            to { opacity: 1; transform: translateY(0); }
        }

        /* Radio buttons */
        .form-check-input {
            background-color: #1a1a1a;
            border-color: #1a3a6e;
        }

        .form-check-input:checked {
            background-color: #1e90ff;
            border-color: #1e90ff;
        }

        .form-check-label { color: #e0e0e0; }

        /* Modal */
        .modal-content {
            background: #0a0a0a;
            border: 1px solid #1e90ff;
        }

        .modal-header {
            border-bottom: 1px solid #1a3a6e;
        }

        .modal-footer {
            border-top: 1px solid #1a3a6e;
        }

        .btn-close {
            filter: invert(1);
        }
    </style>
</head>
<body>
    <nav class="navbar navbar-expand-lg sticky-top">
        <div class="container">
            <a class="navbar-brand" href="/">
                <i class="fas fa-graduation-cap"></i> PlacementPro
            </a>
            <button class="navbar-toggler" type="button" data-bs-toggle="collapse" data-bs-target="#navbarNav">
                <span class="navbar-toggler-icon"></span>
            </button>
            <div class="collapse navbar-collapse" id="navbarNav">
                <ul class="navbar-nav ms-auto">
                    {% if session.get('admin') %}
                        <li class="nav-item"><a class="nav-link" href="/admin/dashboard"><i class="fas fa-tachometer-alt"></i> Dashboard</a></li>
                        <li class="nav-item"><a class="nav-link" href="/admin/manage-students"><i class="fas fa-users"></i> Manage Students</a></li>
                        <li class="nav-item"><a class="nav-link" href="/admin/manage-questions"><i class="fas fa-database"></i> Manage Questions</a></li>
                        <li class="nav-item"><a class="nav-link" href="/logout"><i class="fas fa-sign-out-alt"></i> Logout</a></li>
                    {% elif session.get('student_id') %}
                        <li class="nav-item"><a class="nav-link" href="/student/dashboard"><i class="fas fa-tachometer-alt"></i> Dashboard</a></li>
                        <li class="nav-item"><a class="nav-link" href="/student/exam-select"><i class="fas fa-pen"></i> Take Exam</a></li>
                        <li class="nav-item"><a class="nav-link" href="/student/result"><i class="fas fa-chart-line"></i> Results</a></li>
                        <li class="nav-item"><a class="nav-link" href="/logout"><i class="fas fa-sign-out-alt"></i> Logout</a></li>
                    {% endif %}
                </ul>
            </div>
        </div>
    </nav>

    <div class="container mt-4">
        {% with messages = get_flashed_messages(with_categories=true) %}
            {% if messages %}
                {% for category, message in messages %}
                    <div class="alert alert-{{ category }} alert-dismissible fade show" role="alert">
                        {{ message }}
                        <button type="button" class="btn-close" data-bs-dismiss="alert"></button>
                    </div>
                {% endfor %}
            {% endif %}
        {% endwith %}

        {% block content %}{% endblock %}
    </div>

    <footer>
        <div class="container">
            <p>&copy; 2024 PlacementPro - Smart Placement System</p>
            <small>Skill-Based Assessment | No CGPA Bias</small>
        </div>
    </footer>

    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.1.3/dist/js/bootstrap.bundle.min.js"></script>
</body>
</html>
'''

# Role Selection
ROLE_SELECTION = '''
{% extends "layout.html" %}
{% block content %}
<div class="row justify-content-center align-items-center fade-in" style="min-height: 70vh;">
    <div class="col-12 text-center mb-5">
        <h1 class="display-4 fw-bold" style="color: #1e90ff;">Smart Placement System</h1>
        <p class="lead" style="color: #cccccc;">Select your role to continue</p>
    </div>
    <div class="col-md-5 mb-4">
        <div class="card text-center p-4" onclick="location.href='/admin/login'" style="cursor:pointer">
            <div class="card-body">
                <i class="fas fa-user-shield fa-4x" style="color: #1e90ff;"></i>
                <h3 class="mt-3" style="color: #1e90ff;">Admin</h3>
                <p style="color: #cccccc;">Manage exams, questions, and results</p>
                <button class="btn btn-primary mt-2">Login / Register</button>
            </div>
        </div>
    </div>
    <div class="col-md-5 mb-4">
        <div class="card text-center p-4" onclick="location.href='/student/register'" style="cursor:pointer">
            <div class="card-body">
                <i class="fas fa-user-graduate fa-4x" style="color: #1e90ff;"></i>
                <h3 class="mt-3" style="color: #1e90ff;">Student</h3>
                <p style="color: #cccccc;">Register, take exam, check results</p>
                <button class="btn btn-primary mt-2">Student Portal</button>
            </div>
        </div>
    </div>
</div>
{% endblock %}
'''

# Admin Registration
ADMIN_REGISTER = '''
{% extends "layout.html" %}
{% block content %}
<div class="row justify-content-center fade-in">
    <div class="col-md-5">
        <div class="card">
            <div class="card-header"><h4><i class="fas fa-user-plus"></i> Admin Registration</h4></div>
            <div class="card-body">
                <form method="POST">
                    <div class="mb-3"><label class="form-label">Username</label><input type="text" name="username" class="form-control" required></div>
                    <div class="mb-3"><label class="form-label">Email</label><input type="email" name="email" class="form-control" required></div>
                    <div class="mb-3"><label class="form-label">Password</label><input type="password" name="password" class="form-control" required></div>
                    <button type="submit" class="btn btn-primary w-100">Register</button>
                </form>
                <div class="text-center mt-3"><small>Already have an account? <a href="/admin/login">Login here</a></small></div>
            </div>
        </div>
    </div>
</div>
{% endblock %}
'''

# Admin Login
ADMIN_LOGIN = '''
{% extends "layout.html" %}
{% block content %}
<div class="row justify-content-center fade-in">
    <div class="col-md-5">
        <div class="card">
            <div class="card-header"><h4><i class="fas fa-lock"></i> Admin Login</h4></div>
            <div class="card-body">
                <form method="POST">
                    <div class="mb-3"><label class="form-label">Username or Email</label><input type="text" name="username" class="form-control" required></div>
                    <div class="mb-3"><label class="form-label">Password</label><input type="password" name="password" class="form-control" required></div>
                    <button type="submit" class="btn btn-primary w-100">Login</button>
                </form>
                <div class="text-center mt-3"><small>New admin? <a href="/admin/register">Register here</a></small></div>
                <div class="text-center mt-2"><small>Demo: admin / admin123</small></div>
            </div>
        </div>
    </div>
</div>
{% endblock %}
'''

# Admin Dashboard
ADMIN_DASHBOARD = '''
{% extends "layout.html" %}
{% block content %}
<div class="fade-in">
    <div class="row">
        <div class="col-md-3 mb-3"><div class="stat-card"><i class="fas fa-users fa-2x"></i><div class="stat-number">{{ stats.total_students }}</div><p>Total Students</p></div></div>
        <div class="col-md-3 mb-3"><div class="stat-card" style="border-color: #00ff88;"><i class="fas fa-trophy fa-2x" style="color: #00ff88;"></i><div class="stat-number" style="color: #00ff88;">{{ stats.selected }}</div><p>Selected</p></div></div>
        <div class="col-md-3 mb-3"><div class="stat-card" style="border-color: #ffaa00;"><i class="fas fa-question-circle fa-2x" style="color: #ffaa00;"></i><div class="stat-number" style="color: #ffaa00;">{{ stats.total_questions }}</div><p>Questions</p></div></div>
        <div class="col-md-3 mb-3"><div class="stat-card" style="border-color: #ff4444;"><i class="fas fa-file-pdf fa-2x" style="color: #ff4444;"></i><div class="stat-number" style="color: #ff4444;">{{ stats.pdf_uploads }}</div><p>PDF Uploads</p></div></div>
    </div>

    <div class="row mt-4">
        <div class="col-md-6"><div class="card"><div class="card-header"><h5><i class="fas fa-file-pdf"></i> Upload TECH Exam PDF</h5></div><div class="card-body"><form method="POST" action="/admin/upload-pdf" enctype="multipart/form-data"><input type="hidden" name="category" value="TECH"><div class="mb-3"><input type="file" name="pdf_file" class="form-control" accept=".pdf" required></div><button type="submit" class="btn btn-primary w-100">Convert PDF to TECH MCQ</button></form></div></div></div>
        <div class="col-md-6"><div class="card"><div class="card-header"><h5><i class="fas fa-file-pdf"></i> Upload NON-TECH Exam PDF</h5></div><div class="card-body"><form method="POST" action="/admin/upload-pdf" enctype="multipart/form-data"><input type="hidden" name="category" value="NON-TECH"><div class="mb-3"><input type="file" name="pdf_file" class="form-control" accept=".pdf" required></div><button type="submit" class="btn btn-success w-100">Convert PDF to NON-TECH MCQ</button></form></div></div></div>
    </div>

    <div class="row mt-4">
        <div class="col-md-12"><div class="card"><div class="card-header"><h5><i class="fas fa-download"></i> Download Reports</h5></div><div class="card-body"><a href="/admin/download-all-students" class="btn btn-primary me-2"><i class="fas fa-download"></i> All Students</a><a href="/admin/download-all-results" class="btn btn-success me-2"><i class="fas fa-download"></i> All Results</a><a href="/admin/manage-students" class="btn btn-warning me-2"><i class="fas fa-users"></i> Manage Students</a><a href="/admin/manage-questions" class="btn btn-danger"><i class="fas fa-database"></i> Manage Questions</a></div></div></div>
    </div>

    <div class="card mt-4"><div class="card-header"><h5><i class="fas fa-chart-bar"></i> Question Bank Stats</h5></div><div class="card-body"><p><strong>TECH Questions:</strong> {{ stats.tech_questions }}</p><p><strong>NON-TECH Questions:</strong> {{ stats.nontech_questions }}</p><p><strong>Passing Score:</strong> {{ passing_score }}%</p></div></div>
</div>
{% endblock %}
'''

# Manage Students
MANAGE_STUDENTS = '''
{% extends "layout.html" %}
{% block content %}
<div class="card fade-in"><div class="card-header"><h4><i class="fas fa-users"></i> Manage Students</h4></div><div class="card-body">
<div class="table-responsive"><table class="table table-striped"><thead><tr><th>ID</th><th>Name</th><th>Reg No</th><th>Email</th><th>CGPA</th><th>Hackathons</th><th>Action</th></tr></thead><tbody>
{% for s in students %}
<tr><td>{{ s.student_id }}</td><td>{{ s.name }}</td><td>{{ s.reg_no }}</td><td>{{ s.email }}</td><td>{{ s.cgpa }}</td><td>{{ s.hackathons }}</td>
<td><a href="/admin/delete-student/{{ s.student_id }}" class="delete-btn" onclick="return confirm('Delete this student?')"><i class="fas fa-trash"></i> Delete</a></td></tr>
{% endfor %}
</tbody></table></div>
</div></div>
{% endblock %}
'''

# Manage Questions
MANAGE_QUESTIONS = '''
{% extends "layout.html" %}
{% block content %}
<div class="card fade-in"><div class="card-header"><h4><i class="fas fa-database"></i> Manage Questions</h4></div><div class="card-body">
<ul class="nav nav-tabs"><li class="nav-item"><button class="nav-link active" data-bs-toggle="tab" data-bs-target="#tech">TECH Questions</button></li><li class="nav-item"><button class="nav-link" data-bs-toggle="tab" data-bs-target="#nontech">NON-TECH Questions</button></li></ul>
<div class="tab-content mt-3"><div class="tab-pane active" id="tech"><div class="table-responsive"><table class="table table-striped"><thead><tr><th>ID</th><th>Question</th><th>Options</th><th>Answer</th><th>Action</th></tr></thead><tbody>
{% for q in tech_questions %}
<tr><td>{{ q.question_id }}</td><td>{{ q.question[:80] }}...</td><td>A:{{ q.option_a[:20] }}<br>B:{{ q.option_b[:20] }}</td><td>{{ q.answer }}</td>
<td><a href="/admin/delete-question/{{ q.question_id }}" class="delete-btn" onclick="return confirm('Delete this question?')"><i class="fas fa-trash"></i> Delete</a></td></tr>
{% endfor %}
</tbody></table></div></div>
<div class="tab-pane" id="nontech"><div class="table-responsive"><table class="table table-striped"><thead><tr><th>ID</th><th>Question</th><th>Options</th><th>Answer</th><th>Action</th></tr></thead><tbody>
{% for q in nontech_questions %}
<tr><td>{{ q.question_id }}</td><td>{{ q.question[:80] }}...</td><td>A:{{ q.option_a[:20] }}<br>B:{{ q.option_b[:20] }}</td><td>{{ q.answer }}</td>
<td><a href="/admin/delete-question/{{ q.question_id }}" class="delete-btn" onclick="return confirm('Delete this question?')"><i class="fas fa-trash"></i> Delete</a></td></tr>
{% endfor %}
</tbody></table></div></div></div>
</div></div>
{% endblock %}
'''

# Student Register
STUDENT_REGISTER = '''
{% extends "layout.html" %}
{% block content %}
<div class="row justify-content-center fade-in"><div class="col-md-8"><div class="card"><div class="card-header"><h4><i class="fas fa-user-plus"></i> Student Registration</h4></div><div class="card-body"><form method="POST">
<div class="row"><div class="col-md-6 mb-3"><label class="form-label">Full Name</label><input type="text" name="name" class="form-control" required></div>
<div class="col-md-6 mb-3"><label class="form-label">Register Number</label><input type="text" name="reg_no" class="form-control" required></div>
<div class="col-md-6 mb-3"><label class="form-label">Email</label><input type="email" name="email" class="form-control" required></div>
<div class="col-md-6 mb-3"><label class="form-label">Phone</label><input type="tel" name="phone" class="form-control" required></div>
<div class="col-md-6 mb-3"><label class="form-label">CGPA (0-10)</label><input type="number" step="0.01" name="cgpa" class="form-control" required></div>
<div class="col-md-6 mb-3"><label class="form-label">Department</label><select name="department" class="form-select"><option>CSE</option><option>IT</option><option>ECE</option><option>EEE</option><option>MECH</option></select></div>
<div class="col-md-6 mb-3"><label class="form-label">Skills</label><input type="text" name="skills" class="form-control" required></div>
<div class="col-md-6 mb-3"><label class="form-label">Projects</label><input type="text" name="projects" class="form-control"></div>
<div class="col-md-6 mb-3"><label class="form-label">Certifications</label><input type="text" name="certifications" class="form-control"></div>
<div class="col-md-6 mb-3"><label class="form-label">🏆 Number of Hackathons</label><input type="number" name="hackathons" class="form-control" min="0" value="0" required></div>
<div class="col-md-12 mb-3"><label class="form-label">Password</label><input type="password" name="password" class="form-control" required></div></div>
<button type="submit" class="btn btn-primary w-100">Register</button></form></div></div></div></div>
{% endblock %}
'''

# Student Login
STUDENT_LOGIN = '''
{% extends "layout.html" %}
{% block content %}
<div class="row justify-content-center fade-in"><div class="col-md-5"><div class="card"><div class="card-header"><h4><i class="fas fa-sign-in-alt"></i> Student Login</h4></div><div class="card-body"><form method="POST"><div class="mb-3"><label class="form-label">Student ID or Email</label><input type="text" name="username" class="form-control" required></div><div class="mb-3"><label class="form-label">Password</label><input type="password" name="password" class="form-control" required></div><button type="submit" class="btn btn-primary w-100">Login</button></form><div class="text-center mt-3"><small>New? <a href="/student/register">Register here</a></small></div></div></div></div></div>
{% endblock %}
'''

# Student Dashboard
STUDENT_DASHBOARD = '''
{% extends "layout.html" %}
{% block content %}
<div class="fade-in"><div class="row"><div class="col-md-4"><div class="stat-card"><i class="fas fa-user fa-2x"></i><h4 class="mt-2" style="color:#1e90ff">{{ student.name }}</h4><p>{{ student.reg_no }}</p><p>🏆 Hackathons: {{ student.hackathons }}</p></div></div>
<div class="col-md-8"><div class="card"><div class="card-body"><h5 style="color:#1e90ff">Profile Information</h5><hr><div class="row"><div class="col-md-6"><p><strong>Email:</strong> {{ student.email }}</p><p><strong>CGPA:</strong> {{ student.cgpa }}</p><p><strong>Department:</strong> {{ student.department }}</p></div><div class="col-md-6"><p><strong>Skills:</strong> {{ student.skills }}</p><p><strong>Projects:</strong> {{ student.projects }}</p><p><strong>Certifications:</strong> {{ student.certifications }}</p></div></div></div></div></div></div>
<div class="row mt-4"><div class="col-md-6"><div class="card"><div class="card-body text-center"><h5><i class="fas fa-pen"></i> Take Exam</h5><a href="/student/exam-select" class="btn btn-primary mt-2">Start Exam</a></div></div></div>
<div class="col-md-6"><div class="card"><div class="card-body text-center"><h5><i class="fas fa-chart-line"></i> My Results</h5><a href="/student/result" class="btn btn-info mt-2">View Results</a></div></div></div></div>
{% if result %}<div class="card mt-4"><div class="card-header"><h5>Latest Exam Result</h5></div><div class="card-body text-center"><h2 class="{% if result.status=='SELECTED' %}text-success{% else %}text-warning{% endif %}">{{ "%.2f"|format(result.final_score) }}%</h2><h4><span class="badge bg-{{ 'success' if result.status=='SELECTED' else 'warning' }}">{{ result.status }}</span></h4></div></div>{% endif %}</div>
{% endblock %}
'''

# Exam Select
EXAM_SELECT = '''
{% extends "layout.html" %}
{% block content %}
<div class="row justify-content-center fade-in"><div class="col-md-6"><div class="card"><div class="card-body text-center"><h3 style="color:#1e90ff"><i class="fas fa-clipboard-list"></i> Select Exam Type</h3><hr>
<form method="POST" action="/student/start-exam">
    <button type="submit" name="exam_type" value="TECH" class="btn btn-primary btn-lg m-3" style="min-width:200px"><i class="fas fa-laptop-code"></i> Technical Exam</button>
    <button type="submit" name="exam_type" value="NON-TECH" class="btn btn-success btn-lg m-3" style="min-width:200px"><i class="fas fa-brain"></i> Non-Technical Exam</button>
</form>
<hr><p class="text-muted">Available: TECH: {{ tech_count }} | NON-TECH: {{ nontech_count }}</p></div></div></div></div>
{% endblock %}
'''

# Exam Page
EXAM_PAGE = '''
{% extends "layout.html" %}
{% block content %}
<div class="card fade-in"><div class="card-header"><h4><i class="fas fa-pen-alt"></i> {{ exam_type }} Examination</h4></div><div class="card-body">
<div class="alert alert-info text-center timer" id="timer">⏰ Time Remaining: 30:00</div>
<div class="progress mb-3"><div class="progress-bar" id="progressBar" style="width:0%"></div></div>
<form method="POST" action="/student/submit-exam">
    <input type="hidden" name="exam_type" value="{{ exam_type }}">
    <input type="hidden" name="total_questions" value="{{ total }}">
    {% for q in questions %}
    <div id="q{{ loop.index0 }}" style="display: {% if loop.index0==0 %}block{% else %}none{% endif %};">
        <h5>Question {{ loop.index }} of {{ total }}</h5>
        <p class="lead mt-3">{{ q.question }}</p>
        <div class="options mt-4">
            <div class="form-check mb-2"><input type="radio" name="q{{ q.question_id }}" value="A" id="q{{ q.question_id }}_A"> <label class="form-check-label" for="q{{ q.question_id }}_A"><strong>A.</strong> {{ q.option_a }}</label></div>
            <div class="form-check mb-2"><input type="radio" name="q{{ q.question_id }}" value="B" id="q{{ q.question_id }}_B"> <label class="form-check-label" for="q{{ q.question_id }}_B"><strong>B.</strong> {{ q.option_b }}</label></div>
            <div class="form-check mb-2"><input type="radio" name="q{{ q.question_id }}" value="C" id="q{{ q.question_id }}_C"> <label class="form-check-label" for="q{{ q.question_id }}_C"><strong>C.</strong> {{ q.option_c }}</label></div>
            <div class="form-check mb-2"><input type="radio" name="q{{ q.question_id }}" value="D" id="q{{ q.question_id }}_D"> <label class="form-check-label" for="q{{ q.question_id }}_D"><strong>D.</strong> {{ q.option_d }}</label></div>
        </div>
        <div class="mt-4">
            {% if loop.index0 > 0 %}<button type="button" class="btn btn-secondary" onclick="prevQ({{ loop.index0 }})">Previous</button>{% endif %}
            {% if loop.index0 < total-1 %}<button type="button" class="btn btn-primary" onclick="nextQ({{ loop.index0 }})">Next</button>{% endif %}
            {% if loop.index0 == total-1 %}<button type="button" class="btn btn-success" onclick="submitExam()">Submit Exam</button>{% endif %}
        </div>
    </div>
    {% endfor %}
</form>
</div></div>
<script>
    let timeLeft = 30 * 60, total = {{ total }};
    function updateTimer(){let min=Math.floor(timeLeft/60),sec=timeLeft%60;document.getElementById('timer').innerHTML=`⏰ Time Remaining: ${min}:${sec.toString().padStart(2,'0')}`;if(timeLeft<=0)document.querySelector('form').submit();timeLeft--;}
    setInterval(updateTimer,1000);
    function nextQ(curr){document.getElementById(`q${curr}`).style.display='none';document.getElementById(`q${curr+1}`).style.display='block';updateProgress();}
    function prevQ(curr){document.getElementById(`q${curr}`).style.display='none';document.getElementById(`q${curr-1}`).style.display='block';updateProgress();}
    function updateProgress(){let answered=document.querySelectorAll('input[type="radio"]:checked').length;let pct=(answered/total)*100;document.getElementById('progressBar').style.width=pct+'%';document.getElementById('progressBar').innerHTML=Math.round(pct)+'%';}
    function submitExam(){if(confirm('Submit exam?'))document.querySelector('form').submit();}
</script>
{% endblock %}
'''

# Results Page
RESULTS_PAGE = '''
{% extends "layout.html" %}
{% block content %}
<div class="card text-center fade-in"><div class="card-header {% if result.status=='SELECTED' %}bg-success{% elif result.status=='AVERAGE' %}bg-warning{% else %}bg-danger{% endif %}"><h3>Exam Results</h3></div><div class="card-body">
<div class="row"><div class="col-md-6"><h4>Your Score</h4><h1 class="display-3 {% if result.status=='SELECTED' %}text-success{% elif result.status=='AVERAGE' %}text-warning{% else %}text-danger{% endif %}">{{ "%.2f"|format(result.final_score) }}%</h1></div><div class="col-md-6"><h4>Status</h4><h2><span class="badge bg-{{ 'success' if result.status=='SELECTED' else 'warning' if result.status=='AVERAGE' else 'danger' }} p-3">{{ result.status }}</span></h2></div></div><hr>
<div class="row mt-4"><div class="col-md-4"><div class="card bg-dark"><div class="card-body"><h5>Exam Score</h5><h3>{{ "%.2f"|format(result.exam_score) }}%</h3><small>{{ result.correct_answers }}/{{ result.total_questions }} correct</small></div></div></div>
<div class="col-md-4"><div class="card bg-dark"><div class="card-body"><h5>Project Score</h5><h3>{{ "%.2f"|format(result.project_score) }}%</h3></div></div></div>
<div class="col-md-4"><div class="card bg-dark"><div class="card-body"><h5>Hackathon Bonus</h5><h3>{{ "%.2f"|format(result.hackathon_score) }}%</h3></div></div></div></div>
<div class="mt-4"><a href="/student/dashboard" class="btn btn-primary">Back to Dashboard</a><a href="/student/exam-select" class="btn btn-success ms-2">Take Another Exam</a></div>
</div></div>
{% endblock %}
'''

# Save templates
templates = {
    'layout.html': LAYOUT, 'index.html': ROLE_SELECTION,
    'admin_register.html': ADMIN_REGISTER, 'admin_login.html': ADMIN_LOGIN,
    'admin_dashboard.html': ADMIN_DASHBOARD, 'manage_students.html': MANAGE_STUDENTS,
    'manage_questions.html': MANAGE_QUESTIONS, 'student_register.html': STUDENT_REGISTER,
    'student_login.html': STUDENT_LOGIN, 'student_dashboard.html': STUDENT_DASHBOARD,
    'exam_select.html': EXAM_SELECT, 'exam_page.html': EXAM_PAGE, 'results_page.html': RESULTS_PAGE
}

for filename, content in templates.items():
    with open(f'templates/{filename}', 'w') as f:
        f.write(content)

# ============================================
# ROUTES
# ============================================

@app.route('/')
def index():
    session.clear()
    return render_template_string(ROLE_SELECTION)

# Admin Routes
@app.route('/admin/register', methods=['GET', 'POST'])
def admin_register():
    if request.method == 'POST':
        username = request.form['username']
        email = request.form['email']
        password = request.form['password']
        admins = pd.read_csv('data/admins.csv')
        if username in admins['username'].values:
            return '<div class="container mt-5"><div class="alert alert-danger">Username already exists!</div></div>'
        new_id = f"ADM{len(admins)+1:03d}"
        new_admin = pd.DataFrame([{
            'admin_id': new_id, 'username': username,
            'password': hashlib.md5(password.encode()).hexdigest(),
            'email': email, 'created_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }])
        admins = pd.concat([admins, new_admin], ignore_index=True)
        admins.to_csv('data/admins.csv', index=False)
        return f'<div class="container mt-5"><div class="alert alert-success"><h4>Admin Registered!</h4><p>Username: {username}</p><a href="/admin/login">Login</a></div></div>'
    return render_template_string(ADMIN_REGISTER)

@app.route('/admin/login', methods=['GET', 'POST'])
def admin_login():
    if request.method == 'POST':
        username = request.form['username']
        password = hashlib.md5(request.form['password'].encode()).hexdigest()
        admins = pd.read_csv('data/admins.csv')
        admin = admins[(admins['username'] == username) | (admins['email'] == username)]
        admin = admin[admin['password'] == password]
        if len(admin) > 0:
            session['admin'] = True
            session['admin_username'] = admin.iloc[0]['username']
            return redirect(url_for('admin_dashboard'))
        return '<div class="alert alert-danger">Invalid credentials!</div>'
    return render_template_string(ADMIN_LOGIN)

@app.route('/admin/dashboard')
def admin_dashboard():
    if not session.get('admin'):
        return redirect(url_for('admin_login'))
    students = pd.read_csv('data/students.csv')
    results = pd.read_csv('data/results.csv') if os.path.exists('data/results.csv') else pd.DataFrame()
    questions = pd.read_csv('data/questions.csv') if os.path.exists('data/questions.csv') else pd.DataFrame()
    settings = pd.read_csv('data/settings.csv')
    stats = {
        'total_students': len(students), 'selected': len(results[results['final_score'] >= 80]) if len(results) > 0 else 0,
        'total_questions': len(questions), 'tech_questions': len(questions[questions['category'] == 'TECH']) if len(questions) > 0 else 0,
        'nontech_questions': len(questions[questions['category'] == 'NON-TECH']) if len(questions) > 0 else 0,
        'pdf_uploads': len([f for f in os.listdir('uploads') if f.endswith('.pdf')]) if os.path.exists('uploads') else 0
    }
    passing_score = int(settings[settings['setting_name'] == 'passing_score']['setting_value'].values[0])
    return render_template_string(ADMIN_DASHBOARD, stats=stats, passing_score=passing_score)

@app.route('/admin/manage-students')
def manage_students():
    if not session.get('admin'):
        return redirect(url_for('admin_login'))
    students = pd.read_csv('data/students.csv')
    return render_template_string(MANAGE_STUDENTS, students=students.to_dict('records'))

@app.route('/admin/delete-student/<student_id>')
def delete_student(student_id):
    if not session.get('admin'):
        return redirect(url_for('admin_login'))
    students = pd.read_csv('data/students.csv')
    students = students[students['student_id'] != student_id]
    students.to_csv('data/students.csv', index=False)
    return redirect(url_for('manage_students'))

@app.route('/admin/manage-questions')
def manage_questions():
    if not session.get('admin'):
        return redirect(url_for('admin_login'))
    questions = pd.read_csv('data/questions.csv') if os.path.exists('data/questions.csv') else pd.DataFrame()
    tech_questions = questions[questions['category'] == 'TECH'].to_dict('records') if len(questions) > 0 else []
    nontech_questions = questions[questions['category'] == 'NON-TECH'].to_dict('records') if len(questions) > 0 else []
    return render_template_string(MANAGE_QUESTIONS, tech_questions=tech_questions, nontech_questions=nontech_questions)

@app.route('/admin/delete-question/<int:qid>')
def delete_question(qid):
    if not session.get('admin'):
        return redirect(url_for('admin_login'))
    questions = pd.read_csv('data/questions.csv')
    questions = questions[questions['question_id'] != qid]
    questions.to_csv('data/questions.csv', index=False)
    return redirect(url_for('manage_questions'))

@app.route('/admin/upload-pdf', methods=['POST'])
def upload_pdf():
    if not session.get('admin'):
        return redirect(url_for('admin_login'))
    file = request.files['pdf_file']
    category = request.form['category']
    if file.filename == '':
        return 'No file selected'
    filename = f"{category}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
    filepath = os.path.join('uploads', filename)
    file.save(filepath)
    questions = extract_mcq_from_pdf(filepath)
    existing = pd.read_csv('data/questions.csv') if os.path.exists('data/questions.csv') else pd.DataFrame()
    new_questions = []
    for q in questions:
        new_id = len(existing) + len(new_questions) + 1
        new_questions.append({
            'question_id': new_id, 'question': q['question'],
            'option_a': q.get('option_a', 'Option A'), 'option_b': q.get('option_b', 'Option B'),
            'option_c': q.get('option_c', 'Option C'), 'option_d': q.get('option_d', 'Option D'),
            'answer': q.get('answer', 'A'), 'category': category, 'difficulty': 'Medium',
            'created_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        })
    new_df = pd.DataFrame(new_questions)
    updated = pd.concat([existing, new_df], ignore_index=True)
    updated.to_csv('data/questions.csv', index=False)
    return f'<div class="container mt-5"><div class="alert alert-success"><h4>✅ Added {len(new_questions)} questions!</h4><a href="/admin/dashboard">Back</a></div></div>'

@app.route('/admin/download-all-students')
def download_all_students():
    if not session.get('admin'):
        return redirect(url_for('admin_login'))
    students = pd.read_csv('data/students.csv')
    filename = f"all_students_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
    students.to_excel(filename, index=False, engine='openpyxl')
    return send_file(filename, as_attachment=True)

@app.route('/admin/download-all-results')
def download_all_results():
    if not session.get('admin'):
        return redirect(url_for('admin_login'))
    if os.path.exists('data/results.csv'):
        results = pd.read_csv('data/results.csv')
        filename = f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
        results.to_excel(filename, index=False, engine='openpyxl')
        return send_file(filename, as_attachment=True)
    return "No results found"

# Student Routes
@app.route('/student/register', methods=['GET', 'POST'])
def student_register():
    if request.method == 'POST':
        student_id = f"STU{datetime.now().strftime('%Y%m%d%H%M%S')}"
        new_student = {
            'student_id': student_id, 'name': request.form['name'], 'reg_no': request.form['reg_no'],
            'email': request.form['email'], 'phone': request.form['phone'], 'cgpa': float(request.form['cgpa']),
            'department': request.form['department'], 'skills': request.form['skills'],
            'projects': request.form.get('projects', ''), 'certifications': request.form.get('certifications', ''),
            'hackathons': int(request.form.get('hackathons', 0)), 'resume_path': '',
            'password': hashlib.md5(request.form['password'].encode()).hexdigest(),
            'registration_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        students = pd.read_csv('data/students.csv')
        students = pd.concat([students, pd.DataFrame([new_student])], ignore_index=True)
        students.to_csv('data/students.csv', index=False)
        return f'<div class="container mt-5"><div class="alert alert-success"><h4>Registration Successful!</h4><p>Your ID: {student_id}</p><a href="/student/login">Login</a></div></div>'
    return render_template_string(STUDENT_REGISTER)

@app.route('/student/login', methods=['GET', 'POST'])
def student_login():
    if request.method == 'POST':
        username = request.form['username']
        password = hashlib.md5(request.form['password'].encode()).hexdigest()
        students = pd.read_csv('data/students.csv')
        student = students[(students['student_id'] == username) | (students['email'] == username)]
        student = student[student['password'] == password]
        if len(student) > 0:
            session['student_id'] = student.iloc[0]['student_id']
            session['student_name'] = student.iloc[0]['name']
            return redirect(url_for('student_dashboard'))
        return '<div class="alert alert-danger">Invalid credentials</div>'
    return render_template_string(STUDENT_LOGIN)

@app.route('/student/dashboard')
def student_dashboard():
    if not session.get('student_id'):
        return redirect(url_for('student_login'))
    students = pd.read_csv('data/students.csv')
    student = students[students['student_id'] == session['student_id']].iloc[0].to_dict()
    results = pd.read_csv('data/results.csv') if os.path.exists('data/results.csv') else pd.DataFrame()
    result = results[results['student_id'] == session['student_id']]
    result = result.iloc[-1].to_dict() if len(result) > 0 else None
    return render_template_string(STUDENT_DASHBOARD, student=student, result=result)

@app.route('/student/exam-select', methods=['GET'])
def exam_select_page():
    if not session.get('student_id'):
        return redirect(url_for('student_login'))
    questions = pd.read_csv('data/questions.csv') if os.path.exists('data/questions.csv') else pd.DataFrame()
    tech_count = len(questions[questions['category'] == 'TECH'])
    nontech_count = len(questions[questions['category'] == 'NON-TECH'])
    return render_template_string(EXAM_SELECT, tech_count=tech_count, nontech_count=nontech_count)

@app.route('/student/start-exam', methods=['POST'])
def start_exam():
    if not session.get('student_id'):
        return redirect(url_for('student_login'))
    exam_type = request.form['exam_type']
    questions = pd.read_csv('data/questions.csv')
    exam_questions = questions[questions['category'] == exam_type]
    if len(exam_questions) == 0:
        return f'<div class="container mt-5"><div class="alert alert-warning">No {exam_type} questions available!</div><a href="/student/dashboard">Back</a></div>'
    num_questions = min(20, len(exam_questions))
    selected = exam_questions.sample(n=num_questions).to_dict('records')
    session['exam'] = {'questions': selected, 'exam_type': exam_type}
    return render_template_string(EXAM_PAGE, questions=selected, total=num_questions, exam_type=exam_type)

@app.route('/student/submit-exam', methods=['POST'])
def submit_exam():
    if not session.get('exam'):
        return redirect(url_for('student_dashboard'))
    exam_data = session['exam']
    questions = exam_data['questions']
    correct = 0
    for q in questions:
        answer = request.form.get(f'q{q["question_id"]}')
        if answer and answer == q['answer']:
            correct += 1
    exam_score = (correct / len(questions)) * 100
    students = pd.read_csv('data/students.csv')
    student = students[students['student_id'] == session['student_id']].iloc[0]
    project_score = 70
    hackathon_score = min(20, int(student.get('hackathons', 0)) * 2)
    final_score = (exam_score * 0.7) + (project_score * 0.2) + (hackathon_score * 0.1)
    settings = pd.read_csv('data/settings.csv')
    passing_score = int(settings[settings['setting_name'] == 'passing_score']['setting_value'].values[0])
    if final_score >= passing_score:
        status = "SELECTED"
    elif final_score >= 60:
        status = "AVERAGE"
    else:
        status = "REJECTED"
    result = {
        'student_id': session['student_id'], 'name': student['name'], 'reg_no': student['reg_no'],
        'email': student['email'], 'exam_type': exam_data['exam_type'], 'exam_score': exam_score,
        'project_score': project_score, 'certification_score': 0, 'hackathon_score': hackathon_score,
        'final_score': final_score, 'status': status, 'correct_answers': correct,
        'wrong_answers': len(questions) - correct, 'total_questions': len(questions),
        'exam_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    results = pd.read_csv('data/results.csv') if os.path.exists('data/results.csv') else pd.DataFrame()
    results = pd.concat([results, pd.DataFrame([result])], ignore_index=True)
    results.to_csv('data/results.csv', index=False)
    session.pop('exam', None)
    return render_template_string(RESULTS_PAGE, result=result)

@app.route('/student/result')
def student_result():
    if not session.get('student_id'):
        return redirect(url_for('student_login'))
    results = pd.read_csv('data/results.csv') if os.path.exists('data/results.csv') else pd.DataFrame()
    student_results = results[results['student_id'] == session['student_id']]
    if len(student_results) > 0:
        return render_template_string(RESULTS_PAGE, result=student_results.iloc[-1].to_dict())
    return '<div class="container mt-5"><div class="alert alert-info">No results found</div><a href="/student/dashboard">Back</a></div>'

@app.route('/logout')
def logout():
    session.clear()
    return redirect(url_for('index'))

# ============================================
# ADD SAMPLE QUESTIONS
# ============================================

def add_sample_questions():
    questions = pd.read_csv('data/questions.csv') if os.path.exists('data/questions.csv') else pd.DataFrame()
    if len(questions) == 0:
        sample = [
            [1, "What is the time complexity of binary search?", "O(n)", "O(log n)", "O(n²)", "O(1)", "B", "TECH", "Easy", datetime.now().strftime('%Y-%m-%d')],
            [2, "Which data structure uses LIFO?", "Queue", "Array", "Stack", "Linked List", "C", "TECH", "Easy", datetime.now().strftime('%Y-%m-%d')],
            [3, "What is 15% of 200?", "20", "25", "30", "35", "C", "NON-TECH", "Easy", datetime.now().strftime('%Y-%m-%d')],
            [4, "Complete: 2, 4, 8, 16, ?", "24", "32", "30", "28", "B", "NON-TECH", "Easy", datetime.now().strftime('%Y-%m-%d')],
            [5, "Choose correct spelling", "Recieve", "Receive", "Reeceive", "Recive", "B", "NON-TECH", "Easy", datetime.now().strftime('%Y-%m-%d')],
        ]
        questions = pd.DataFrame(sample, columns=['question_id', 'question', 'option_a', 'option_b', 'option_c', 'option_d', 'answer', 'category', 'difficulty', 'created_date'])
        questions.to_csv('data/questions.csv', index=False)
        print("✅ Added sample questions!")

add_sample_questions()

# ============================================
# START APPLICATION
# ============================================

def start_app():
    print("\n" + "="*60)
    print("🎉 SMART PLACEMENT SYSTEM READY!")
    print("="*60)
    port = find_free_port()
    if NGROK_AVAILABLE:
        try:
            public_url = ngrok.connect(port)
            print(f"\n🔗 PUBLIC URL: {public_url}")
        except Exception as e:
            print(f"\n⚠️ Ngrok error: {e}")
    print(f"\n📝 Admin Login: admin / admin123")
    print(f"📝 Or register new admin at: /admin/register")
    print(f"🌐 Local URL: http://localhost:{port}")
    print("\n" + "="*60)
    print("🎨 DARK THEME ACTIVE!")
    print("   Background: Black")
    print("   Highlights: Dark Blue (#1e90ff)")
    print("="*60 + "\n")
    app.run(host='0.0.0.0', port=port, debug=False, use_reloader=False)

start_app()

✅ ngrok ready!

🎉 SMART PLACEMENT SYSTEM READY!

🔗 PUBLIC URL: NgrokTunnel: "https://tux-grievance-dandelion.ngrok-free.dev" -> "http://localhost:60937"

📝 Admin Login: admin / admin123
📝 Or register new admin at: /admin/register
🌐 Local URL: http://localhost:60937

🎨 DARK THEME ACTIVE!
   Background: Black
   Highlights: Dark Blue (#1e90ff)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:60937
 * Running on http://172.28.0.12:60937
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [14/Apr/2026 15:16:52] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/Apr/2026 15:17:03] "GET /admin/login HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/Apr/2026 15:17:32] "POST /admin/login HTTP/1.1" 302 -
INFO:werkzeug:127.0.0.1 - - [14/Apr/2026 15:17:33] "GET /admin/dashboard HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/Apr/2026 15:18:26] "POST /admin/upload-pdf HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/Apr/2026 15:19:12] "POST /admin/upload-pdf HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/Apr/2026 15:19:31] "GET /admin/manage-questions HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/Apr/2026 15:19:48] "GET /admin/delete-question/4 HTTP/1.1" 302 -
INFO:werk